# The Score You Get From Tuning on Your Test Set

**Topic:** Nested cross-validation &nbsp;·&nbsp; **Difficulty:** Hard

You try 40 hyperparameter settings, keep the best cross-validated score, and
report it. That number is biased upward: you have selected the maximum of 40
noisy estimates, and the maximum of noisy things is optimistic even when nothing
is learnable.

To make the point unmissable, the labels here are **pure noise**. There is
nothing to learn. Flat CV will still hand you a score above chance, but nested CV
will not.

## The data (created in the next cell)

- `X` `(200, 60)`: random features.
- `y` `(200,)`: random labels, **independent of `X`**. True accuracy is 0.5.
- 40 candidate values of the regularization strength `C`.

## Your task

Using **sklearn** (`LogisticRegression(max_iter=500)`) and
`KFold(n_splits=5, shuffle=True, random_state=0)` for both loops:

1. `flat_best`: for each `C` in `Cs`, compute 5-fold CV accuracy on all of `X` and
   report the **maximum** over `C`.
2. `nested`: outer 5-fold, then inside each outer training set run an inner 5-fold
   over all `C`, pick the best `C` there, refit on the outer training set, and
   score the outer test fold. Average the five outer scores.
3. Set `answers` to a dict with `"flat_best"`, `"nested"`, and `"optimism"` =
   `flat_best - nested`.

Cost: the nested part runs 5 x (40 x 5 + 1) = 1005 small logistic fits on top of
the 200 flat fits, so expect it to take about five times as long as the flat
part. No GPU and no download are needed.

## Evaluation

Compared to the reference within `1e-6 * (1 + |ref|)`. The evaluation cell
checks the flat estimate exceeds the nested one and that nested sits near 0.5.

In [ ]:
import numpy as np

rng = np.random.default_rng(909)
n, d = 200, 60
X = rng.normal(0, 1, (n, d))
y = rng.integers(0, 2, n)               # labels independent of X
Cs = np.logspace(-4, 2, 40)
print(f"X {X.shape}, y is pure noise: true accuracy is 0.500")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("flat_best", [], 0.525, -0.08790495777761206, 1e-06),
    ("nested", [], 0.45, -0.07534710666652462, 1e-06),
    ("optimism", [], 0.07500000000000001, -0.012557851111087438, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "optimism" in answers:
    _o = answers["optimism"] > 0 and abs(answers["nested"] - 0.5) < 0.12
    print(f"  {'flat is optimistic':<24} {'ok' if _o else 'wrong'}  "
          f"(flat {answers['flat_best']:.3f} vs nested {answers['nested']:.3f})")
    _ok = _ok and _o

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

Flat CV reuses the same folds to choose `C` and to report the score, so the
reported number is `max` over 40 correlated noisy estimates, reliably above 0.5
even though `y` is independent of `X`. Nested CV chooses `C` using only the outer
training data, so the outer folds stay untouched by selection and the estimate
lands where it should. Any time you tune and report on the same split, you owe
the reader this caveat.

```python
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=0)

def cv_acc(Xd, yd, C, splitter):
    s = []
    for tr, te in splitter.split(Xd):
        m = LogisticRegression(C=C, max_iter=500).fit(Xd[tr], yd[tr])
        s.append((m.predict(Xd[te]) == yd[te]).mean())
    return float(np.mean(s))

flat = [cv_acc(X, y, C, kf) for C in Cs]
flat_best = float(np.max(flat))

outer = KFold(n_splits=5, shuffle=True, random_state=0)
scores = []
for tr, te in outer.split(X):
    inner = KFold(n_splits=5, shuffle=True, random_state=0)
    inner_scores = [cv_acc(X[tr], y[tr], C, inner) for C in Cs]
    best_C = Cs[int(np.argmax(inner_scores))]
    m = LogisticRegression(C=best_C, max_iter=500).fit(X[tr], y[tr])
    scores.append((m.predict(X[te]) == y[te]).mean())
nested = float(np.mean(scores))

answers = {"flat_best": flat_best, "nested": nested,
           "optimism": flat_best - nested}
```